# 🚀 VCP Configuration Generator

This notebook demonstrates how to programmatically generate configuration files for the **Pionix Virtual Charger Park (VCP)**. 

### 💡 Flexibility of VCP Templates

The VCP platform is designed to be highly flexible. You are **not limited** to a fixed set of headers or formats. 
- **Custom Placeholders**: You can define your own placeholders (e.g., `%MyCustomSetting%`) in your EVerest or OCPP configurations, and then provide values for them in your CSV.
- **Multiple Protocols**: While this example focuses on an OCPP 1.6 setup, the same logic applies to OCPP 2.0.1 or any custom configuration you've built.
- **User-Defined Formats**: You can structure your CSV however you like, as long as the headers match the placeholders used in your VCP templates.

## 📋 Example: OCPP 1.6 Template Format

This is a common example for an OCPP 1.6 configuration. Note that these headers match placeholders defined in a specific VCP template.

| Header | Description |
|---|---|
| `%CpoName%` | Name of the Charge Point Operator. |
| `%EVSE_ID%` | Unique identifier for the EVSE (e.g., `DE*PNX*E1`). |
| `%FirmwareVersion%` | Version string for the EVerest firmware. |
| `%SecurityProfile%` | OCPP Security Profile (0 for Unsecured, 1-3 for TLS/mTLS). |
| `%AuthorizationKey%` | Basic Auth password for the charger. |
| `%ChargePointModel%` | Hardware model identifier. |
| `%OCPP_BACKEND_URI%` | Optional override for the Central System URL. |
| `%ChargePointVendor%` | Name of the hardware manufacturer. |
| `%EVSE_FUSE_LIMIT_A%` | Current limit for the individual EVSE in Amperes. |
| `%STATION_FUSE_LIMIT_A%` | Total current limit for the charging station. |
| `%ChargePointSerialNumber%` | Unique hardware serial number. |

## 🛠️ Generating Variations

The following Python code generates a randomized but **deterministic** CSV file. By using a fixed seed, we ensure that the same configuration is generated every time you run this cell.

In [ ]:
import csv
import random
import os

def generate_vcp_fleet(num_stations=10, filename="vcp_fleet_config.csv", seed=42, backend_uri=""):
    # Set seed for deterministic output
    random.seed(seed)
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(filename), exist_ok=True) if os.path.dirname(filename) else None
    
    # Fictional vendors and models
    vendors = ["CyberCharge", "NebulaGrid", "EcoPulse", "StellarWatt"]
    models = {
        "CyberCharge": ["CC-Alpha", "CC-Pro-X"],
        "NebulaGrid": ["Nova-S", "Nova-Max"],
        "EcoPulse": ["GreenLeaf-22", "EcoBase"],
        "StellarWatt": ["Star-Charge-AC", "Galaxy-DC"]
    }
    firmwares = ["1.2.3", "2.0.0-rc", "1.5.0"]
    
    headers = [
        "%CpoName%", "%EVSE_ID%", "%FirmwareVersion%", "%SecurityProfile%", 
        "%AuthorizationKey%", "%ChargePointModel%", "%OCPP_BACKEND_URI%", 
        "%ChargePointVendor%", "%EVSE_FUSE_LIMIT_A%", "%STATION_FUSE_LIMIT_A%", 
        "%ChargePointSerialNumber%"
    ]

    # Generate a single shared authorization key for all stations (OCPP requires 16-20 chars)
    auth_key = ''.join(random.choices("0123456789abcdef", k=20))

    rows = []
    serial_numbers = []
    
    for i in range(1, num_stations + 1):
        vendor = random.choice(vendors)
        model = random.choice(models[vendor])
        
        # Generate deterministic values using random with seed
        cpo_name = "VCP Test Operations"
        evse_id = f"DE*VCP*E{i:03d}"
        firmware = random.choice(firmwares)
        security_profile = 2
        
        # Logic for fuse limits
        evse_limit = float(random.choice([16, 32, 63]))
        station_limit = evse_limit + random.choice([0, 16])
        
        # Deterministic serial number
        suffix = ''.join(random.choices("ABCDEFGH", k=4))
        serial = f"VCP-{i:03d}-{suffix}"
        serial_numbers.append(serial)
        
        rows.append([
            cpo_name, evse_id, firmware, security_profile, 
            auth_key, model, backend_uri, vendor,
            evse_limit, station_limit, serial
        ])

    # Write to file
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(rows)
    
    print(f"✅ Successfully generated {num_stations} variations in {filename} (Seed: {seed})")
    print(f"🔑 Shared Authorization Key: {auth_key}")
    
    print("\n--- Generated Serial Numbers ---")
    print("\n".join(serial_numbers))
    
    return filename

# --- Main Execution ---

# 1. Ask for Backend URI
try:
    target_backend = input("🔗 Enter your OCPP Backend URI (e.g., your ngrok URL) or leave empty for default: ").strip()
except EOFError:
    # Fallback for non-interactive environments
    target_backend = ""

# 2. Generate a sample fleet
output_file = generate_vcp_fleet(num_stations=50, filename="output/vcp_fleet_config.csv", backend_uri=target_backend)

# 3. Display the first few lines of the CSV
with open(output_file, 'r') as f:
    print("\n--- Preview of generated CSV ---")
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 3: break

## 📤 Next Steps

Once you have generated your `vcp_fleet_config.csv` file, you can:
1.  Open the **Pionix VCP Dashboard**.
2.  Navigate to your OCPP 1.6 **Virtual Charger Park**.
3.  Open **Config Placeholders**.
4.  Upload the generated CSV to apply the configurations.